# Kiểm chứng lựa chọn Loss Function bằng Validation (chống rò rỉ)

**Dự án Tốt nghiệp - Energy Forecasting - Nhóm The Outliers**

## 1. Mục đích

Notebook này **KHÔNG train lại gì cả** — chỉ load lại 6 model `.pkl` đã train sẵn (MAE/Huber/MSE × H1/H4) và tính WAPE trên `v4_val_selected`, tập tách biệt với train và test, để xem model nào thắng thật.

Lý do cần làm: phát hiện lỗi rò rỉ dữ liệu — bước chọn loss function trước đây đang dùng nhầm tập test thay vì validation (`metrics_val.json` bị ghi từ metric tính trên `test_h`). Notebook này verify lại bằng validation thật, không đụng test, chạy trong vài giây.

## 2. Import và cấu hình đường dẫn

In [1]:
import os
import json
import pickle
import numpy as np
import pandas as pd

BASE = '/home/tandat/Desktop/Du_An_Tot_Nghiep_v3'
VAL_PATH = f'{BASE}/data/model/v4/05_selected/v4_val_selected.parquet'
TRAIN_DIR = f'{BASE}/data/model/v4/06_train'
EPS_ELEV = 0.05
SITE_COL = 'site_id'
TIMESTAMP_COL = 'timestamp'
TARGET_COL = 'energy_generated_kwh'

print('Da cau hinh duong dan. VAL_PATH =', VAL_PATH)
print('TRAIN_DIR =', TRAIN_DIR)

Da cau hinh duong dan. VAL_PATH = /home/tandat/Desktop/Du_An_Tot_Nghiep_v3/data/model/v4/05_selected/v4_val_selected.parquet
TRAIN_DIR = /home/tandat/Desktop/Du_An_Tot_Nghiep_v3/data/model/v4/06_train


## 3. Danh sách cột dịch theo horizon (`_mt`)

Sao chép đúng từ `04_1_train_mae.py` (COT_TAT_DINH) — các cột này được dịch tới thời điểm T+h thành cột mới hậu tố `_mt`, giống hệt hàm `them_muc_tieu()` trong pipeline training thật.

In [2]:
COT_TAT_DINH = [
    'solar_elevation', 'solar_azimuth', 'azimuth_sin', 'azimuth_cos', 'sin_elevation',
    'ghi_cs', 'clearsky_proxy', 'ky_vong', 'ty_le_bao_hoa',
    'minute_of_day', 'hour_of_day', 'hour_bucket_model', 'hour', 'hour_sin', 'hour_cos',
    'minute', 'day', 'day_of_week', 'month', 'day_of_year', 'doy_sin', 'doy_cos',
]
print(f'Co {len(COT_TAT_DINH)} cot se duoc dich thanh dac trung _mt.')

Co 22 cot se duoc dich thanh dac trung _mt.


## 4. Hàm chuẩn hoá mục tiêu và tính WAPE

Giống hệt `mau_chuan_hoa()` trong script training gốc.

In [3]:
def mau_chuan_hoa(df):
    """Mau so de chuan hoa muc tieu: quy mo tram nhan sin(goc cao mat troi)."""
    return (df['site_scale'] * np.clip(df['sin_elevation'], EPS_ELEV, None)).to_numpy()


def compute_wape(yt, yp):
    yt = np.asarray(yt, dtype=float)
    yp = np.asarray(yp, dtype=float)
    denom = np.sum(np.abs(yt))
    return float(np.sum(np.abs(yt - yp)) / denom * 100.0) if denom > 0 else float('nan')

print('Da dinh nghia mau_chuan_hoa va compute_wape.')

Da dinh nghia mau_chuan_hoa va compute_wape.


## 5. Nạp tập validation holdout

Đọc trực tiếp `v4_val_selected.parquet`, tự tạo `y_true` (dịch target theo horizon, giống `them_muc_tieu()`) và các cột `_mt` cần thiết. Tập này không dùng để train hoặc test.

In [4]:
# Hai tram 19 va 24 co capacity_kw khong dang tin nen da bi loai khoi train; phai loai
# ca o buoc cham diem, neu khong thi cham tren tram ma mo hinh chua he hoc.
EXCLUDE_SITES = [19, 24]

# Cac cot phan loai can dich sang moc nhan T+h de loc pham vi cho dung.
COT_NHAN = ['energy_source', 'is_daylight']


def load_val_holdout(features, horizon_steps):
    # Can doc them cac cot GOC (khong _mt) cua COT_TAT_DINH de dich thanh _mt.
    base_needed = [c[:-3] if c.endswith('_mt') and c[:-3] in COT_TAT_DINH else c for c in features]
    need = list(dict.fromkeys(
        base_needed + [SITE_COL, TIMESTAMP_COL, TARGET_COL, 'site_scale', 'sin_elevation',
                       'tran_cong_suat', 'energy_source', 'is_daylight']))
    d = pd.read_parquet(VAL_PATH)
    need = [c for c in need if c in d.columns]
    d = d[need].sort_values([SITE_COL, TIMESTAMP_COL]).reset_index(drop=True)

    if EXCLUDE_SITES and SITE_COL in d.columns:
        _n0 = len(d)
        d = d[~d[SITE_COL].isin(EXCLUDE_SITES)].reset_index(drop=True)
        print(f'   Loai site {EXCLUDE_SITES}: {_n0:,} -> {len(d):,} dong')

    h = int(horizon_steps)
    d['y_true'] = d.groupby(SITE_COL)[TARGET_COL].shift(-h)
    g = d.groupby(SITE_COL)
    for c in COT_TAT_DINH:
        if c in d.columns and f'{c}_mt' in features:
            d[f'{c}_mt'] = g[c].shift(-h)

    # FIX: nhan la san luong tai T+h, nen pham vi "do that ban ngay" phai xet DONG NHAN
    # tai T+h chu khong phai dong dac trung tai T. Ban truoc loc theo cot tai T nen van
    # dem ca nhung dong ma NHAN do buoc dien khuyet ETL sinh ra hoac roi vao ban dem.
    # Cung cach ma notebook 06_1 va notebook 07 dang dung (tien to nhan_).
    for c in COT_NHAN:
        if c in d.columns:
            d[f'nhan_{c}'] = g[c].shift(-h)

    d = d.dropna(subset=['y_true'])
    return d[(d['site_scale'] > 0) & (d['sin_elevation'] > EPS_ELEV)].copy()


print('Da dinh nghia load_val_holdout (v4_val_selected, loc theo nhan tai T+h).')

Da dinh nghia load_val_holdout (v4_val_selected, loc theo nhan tai T+h).


## 6. Đánh giá 1 model (load lại `.pkl`, dự báo trên validation, tính metric)

In [5]:
_VAL_CACHE = {}


def _metric_1_scope(yt, yp):
    yt = np.asarray(yt, dtype=float); yp = np.asarray(yp, dtype=float)
    rmse = float(np.sqrt(np.mean((yt - yp) ** 2))) if len(yt) else float('nan')
    mae = float(np.mean(np.abs(yt - yp))) if len(yt) else float('nan')
    ss_res = np.sum((yt - yp) ** 2); ss_tot = np.sum((yt - np.mean(yt)) ** 2)
    r2 = float(1 - ss_res / ss_tot) if len(yt) and ss_tot > 0 else float('nan')
    return {'wape': compute_wape(yt, yp), 'rmse': rmse, 'mae': mae, 'r2': r2, 'n': int(len(yt))}


def eval_one(loss_name, h_label):
    cfg_path = f'{TRAIN_DIR}/{loss_name}/{h_label}/model_config.json'
    pkl_path = f'{TRAIN_DIR}/{loss_name}/{h_label}/model.pkl'
    cfg = json.load(open(cfg_path))
    features = cfg['features']
    medians = cfg['feature_medians']
    horizon_steps = cfg['horizon_steps']

    # FIX (2026-08-16): truoc day o duoi cat k tai hang so 1.5 ghi cung, trong khi ba notebook
    # train (06_1/06_2/06_3) deu cat tai clip_k suy tu phan vi 99 cua k tren tap TRAIN
    # (= 1.3763921048868135). Hai nguong khac nhau -> cung mot mo hinh cho hai bo metric khac
    # nhau, ma chinh notebook nay ghi de metrics_val.json ma bao cao doc. Doc clip_k tu
    # model_config.json giong het cach 07_final_test dang lam, va bao loi neu thieu thay vi
    # am tham dung mac dinh.
    if 'clip_k' not in cfg:
        raise KeyError(
            f'{cfg_path} thieu khoa clip_k. Hay chay lai notebook train de sinh lai config.'
        )
    clip_k = float(cfg['clip_k'])

    with open(pkl_path, 'rb') as f:
        model = pickle.load(f)

    cache_key = int(horizon_steps)
    if cache_key not in _VAL_CACHE:
        _VAL_CACHE[cache_key] = load_val_holdout(features, horizon_steps)
    val = _VAL_CACHE[cache_key]

    missing_feats = [c for c in features if c not in val.columns]
    if missing_feats:
        raise KeyError(f'Thieu {len(missing_feats)} dac trung: {missing_feats[:5]}...')

    X = val[features].fillna(pd.Series(medians)).astype(np.float32)
    k_pred = np.clip(model.predict(X), 0, clip_k)
    y_pred = np.minimum(k_pred * mau_chuan_hoa(val), val['tran_cong_suat'].to_numpy() * 1.02)
    y_pred = np.where(val['sin_elevation'].to_numpy() <= EPS_ELEV, 0.0, y_pred)
    y_true = val['y_true'].to_numpy()

    scope_all = _metric_1_scope(y_true, y_pred)

    # Loc theo cot NHAN tai T+h (tien to nhan_), lui ve cot tai T neu khung khong co.
    def _cot_loc(ten):
        return val[f'nhan_{ten}'] if f'nhan_{ten}' in val.columns else val.get(ten)

    mask = np.ones(len(val), dtype=bool)
    _nguon = _cot_loc('energy_source')
    if _nguon is not None:
        mask &= (_nguon == 'measured').to_numpy()
    _ngay = _cot_loc('is_daylight')
    if _ngay is not None:
        mask &= _ngay.fillna(False).astype(bool).to_numpy()
    scope_md = _metric_1_scope(y_true[mask], y_pred[mask])

    # Ghi metric tren holdout v4_val_selected de notebook 07 doc dung o lan chay tiep theo.
    out_dir = f'{TRAIN_DIR}/{loss_name}/{h_label}'
    with open(f'{out_dir}/metrics_val.json', 'w', encoding='utf-8') as f:
        json.dump({'horizon_steps': int(horizon_steps), 'loss_name': loss_name,
                   'feature_set_name': '', 'measured_daylight': scope_md,
                   'all': scope_all}, f, indent=2, ensure_ascii=False, default=str)

    return {'loss': loss_name, 'horizon': h_label, 'n_rows_val_total': len(val),
            'n_measured_daylight': scope_md['n'], 'wape_val_%': round(scope_md['wape'], 4),
            'rmse_val': round(scope_md['rmse'], 4), 'mae_val': round(scope_md['mae'], 4),
            'r2_val': round(scope_md['r2'], 4)}

print('Da dinh nghia eval_one (v4_val_selected -> metrics_val.json).')

Da dinh nghia eval_one (v4_val_selected -> metrics_val.json).


**Lưu ý:** `eval_one()` ghi `metrics_val.json` sau khi chấm trên `v4_val_selected`, nên notebook 07 đọc đúng holdout validation.

## 7. Chạy đánh giá cho cả 6 model và xác định model thắng thật

In [6]:
rows = []
for loss in ['mae', 'huber', 'mse']:
    for h in ['h1', 'h4']:
        try:
            r = eval_one(loss, h)
            rows.append(r)
            print(f"{loss:6s} {h}: WAPE_val={r['wape_val_%']:>8.4f}%  RMSE_val={r['rmse_val']:>8.4f}  "
                  f"MAE_val={r['mae_val']:>8.4f}  R2_val={r['r2_val']:>7.4f}  "
                  f"(n={r['n_measured_daylight']:,} / {r['n_rows_val_total']:,})")
        except Exception as e:
            print(f'{loss:6s} {h}: LOI - {e}')

df_ket_qua = pd.DataFrame(rows)
display(df_ket_qua)

   Loai site [19, 24]: 723,114 -> 688,680 dong
mae    h1: WAPE_val= 22.0523%  RMSE_val=  3.6737  MAE_val=  1.5062  R2_val= 0.8984  (n=288,811 / 318,981)
   Loai site [19, 24]: 723,114 -> 688,680 dong
mae    h4: WAPE_val= 26.9712%  RMSE_val=  4.3598  MAE_val=  1.9338  R2_val= 0.8628  (n=272,569 / 318,861)
huber  h1: WAPE_val= 22.7937%  RMSE_val=  3.6608  MAE_val=  1.5569  R2_val= 0.8991  (n=288,811 / 318,981)
huber  h4: WAPE_val= 27.4086%  RMSE_val=  4.2896  MAE_val=  1.9651  R2_val= 0.8672  (n=272,569 / 318,861)
mse    h1: WAPE_val= 22.6591%  RMSE_val=  3.6373  MAE_val=  1.5477  R2_val= 0.9004  (n=288,811 / 318,981)
mse    h4: WAPE_val= 27.3365%  RMSE_val=  4.2850  MAE_val=  1.9600  R2_val= 0.8674  (n=272,569 / 318,861)


,loss,horizon,n_rows_val_total,n_measured_daylight,wape_val_%,rmse_val,mae_val,r2_val
0,mae,h1,318981,288811,22.0523,3.6737,1.5062,0.8984
1,mae,h4,318861,272569,26.9712,4.3598,1.9338,0.8628
2,huber,h1,318981,288811,22.7937,3.6608,1.5569,0.8991
3,huber,h4,318861,272569,27.4086,4.2896,1.9651,0.8672
4,mse,h1,318981,288811,22.6591,3.6373,1.5477,0.9004
5,mse,h4,318861,272569,27.3365,4.2850,1.9600,0.8674


In [7]:
print('=== MODEL THANG THAT TREN VALIDATION (WAPE thap nhat, khong dung tap test) ===')
nguoi_thang = {}
for h in ['h1', 'h4']:
    sub = [r for r in rows if r['horizon'] == h]
    if sub:
        best = min(sub, key=lambda r: r['wape_val_%'])
        nguoi_thang[h] = best['loss']
        print(f"{h}: {best['loss'].upper()} thang voi WAPE_val = {best['wape_val_%']:.4f}%")
print()
print('Ket qua nay dung de CHON model, khong dung de bao cao headline.')
print('So lieu headline chinh thuc van lay tu ket_qua.json / metrics_overall.json tren tap TEST')
print('cua dung model vua thang o day (chi doc 1 lan, khong dung de chon lai).')

=== MODEL THANG THAT TREN VALIDATION (WAPE thap nhat, khong dung tap test) ===
h1: MAE thang voi WAPE_val = 22.0523%
h4: MAE thang voi WAPE_val = 26.9712%

Ket qua nay dung de CHON model, khong dung de bao cao headline.
So lieu headline chinh thuc van lay tu ket_qua.json / metrics_overall.json tren tap TEST
cua dung model vua thang o day (chi doc 1 lan, khong dung de chon lai).


## 8. Xuất kết quả ra file (để trích dẫn trong report)

In [8]:
OUT_PATH = f'{BASE}/data/model/v4/07_final_test/val_model_selection_check.json'
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
with open(OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump({'ket_qua_tung_model': rows, 'model_thang': nguoi_thang}, f, indent=2, ensure_ascii=False)
print(f'Da luu: {OUT_PATH}')

Da luu: /home/tandat/Desktop/Du_An_Tot_Nghiep_v3/data/model/v4/07_final_test/val_model_selection_check.json
